# gigapath_runpod (로컬→RunPod SSH 오케스트레이션)

로컬 주피터 노트북에서 SSH/rsync로 RunPod 인스턴스를 제어하며 **SVS 업로드 → 타일링 → 원본 삭제 → 벡터화 → 타일 삭제 → 학습** 순서로 스토리지 사용을 최소화합니다. 아래 자리표시자(타일링/벡터화/학습 커맨드, 호스트명 등)를 환경에 맞게 수정하세요.


## 0. 기본 설정 (호스트/경로/커맨드 자리표시자)

- `SSH_HOST_DIRECT/PORT`: RunPod Direct TCP (SCP/rsync 가능)
- `SSH_HOST_GATEWAY`: 필요 시 게이트웨이 사용 (SCP 제한)
- `REMOTE_RAW`, `REMOTE_WORK`: RunPod 내부 경로
- 타일링/벡터화/학습 커맨드를 gigapath 코드에 맞게 채우기
- 삭제 정책 토글: `DELETE_SVS_AFTER_TILING`, `DELETE_TILES_AFTER_FEATURE`


In [1]:
import subprocess, shlex, time
from pathlib import Path

# SSH 대상 및 키 (실제 RunPod 접속 정보로 설정)
# 게이트웨이(ssh.runpod.io)와 Direct TCP(root@IP -p PORT)를 모두 지원
SSH_HOST_GATEWAY = "crxrrtk7qj79so-64411c13@ssh.runpod.io"  # proxied, SCP 제한
SSH_HOST_DIRECT = "root@216.81.151.15"  # direct TCP, SCP/rsync 가능
SSH_PORT_DIRECT = 13458
SSH_KEY = "~/.ssh/runpod_peter"  # 필요 없으면 None
# 기본은 Direct TCP를 사용(PTY 오류 회피, rsync/ssh 모두 동일 경로)
USE_DIRECT_FOR_SSH = True
USE_DIRECT_FOR_RSYNC = True
# PTY 옵션: direct에서는 비워두고, 게이트웨이 필요 시 설정 (예: "-T")
SSH_EXTRA_OPTS = ""

# RunPod 내부 경로
REMOTE_RAW = "~/data/raw"
REMOTE_WORK = "~/data/work"
REMOTE_CHECKPOINT = f"{REMOTE_WORK}/checkpoints"
REMOTE_LOG = "~/logs"

# 타일링/벡터화/학습 커맨드 템플릿 (gigapath 코드에 맞게 수정)
TILE_SIZE = 224
TILE_OVERLAP = 0
TILING_CMD_TEMPLATE = (
    "python tools/tile_svs.py --input {input} --output {output} "
    "--tile-size {tile_size} --overlap {overlap}"
)
FEATURE_CMD_TEMPLATE = "python tools/extract_features.py --tiles {tiles} --out {out}"
TRAIN_CMD_TEMPLATE = (
    "python train.py --features {features_dir} --out {ckpt_dir} --logdir {log_dir}"
)

# 삭제 정책
DELETE_SVS_AFTER_TILING = True
DELETE_TILES_AFTER_FEATURE = True

print("SSH_HOST_DIRECT=", SSH_HOST_DIRECT, "port", SSH_PORT_DIRECT)
print("SSH_HOST_GATEWAY=", SSH_HOST_GATEWAY)
print("REMOTE_RAW=", REMOTE_RAW)
print("REMOTE_WORK=", REMOTE_WORK)
print("TILING_CMD_TEMPLATE=", TILING_CMD_TEMPLATE)
print("FEATURE_CMD_TEMPLATE=", FEATURE_CMD_TEMPLATE)
print("TRAIN_CMD_TEMPLATE=", TRAIN_CMD_TEMPLATE)


SSH_HOST_DIRECT= root@216.81.151.15 port 13458
SSH_HOST_GATEWAY= crxrrtk7qj79so-64411c13@ssh.runpod.io
REMOTE_RAW= ~/data/raw
REMOTE_WORK= ~/data/work
TILING_CMD_TEMPLATE= python tools/tile_svs.py --input {input} --output {output} --tile-size {tile_size} --overlap {overlap}
FEATURE_CMD_TEMPLATE= python tools/extract_features.py --tiles {tiles} --out {out}
TRAIN_CMD_TEMPLATE= python train.py --features {features_dir} --out {ckpt_dir} --logdir {log_dir}


## 1. 유틸리티: 로컬 실행/SSH/rsync 래퍼

- `run_local`: 로컬 셸 실행
- `run_ssh`: RunPod에서 명령 실행
- `rsync_upload` / `rsync_download`: 부분 업로드/다운로드 지원


In [2]:
def run_local(cmd, check=True):
    print(f"[local] $ {cmd}")
    result = subprocess.run(cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Local command failed: {cmd}")
    return result.returncode

def _ssh_parts(host, port=None):
    parts = ["ssh"]
    if SSH_KEY:
        parts += ["-i", SSH_KEY]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if port:
        parts += ["-p", str(port)]
    parts += [host]
    return parts

def run_ssh(cmd, check=True, use_direct=None):
    if use_direct is None:
        use_direct = USE_DIRECT_FOR_SSH
    parts = _ssh_parts(
        SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if use_direct else None,
    )
    ssh_cmd = " ".join(shlex.quote(p) for p in parts + [cmd])
    print(f"[ssh] $ {cmd}")
    result = subprocess.run(ssh_cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"SSH command failed: {cmd}")
    return result.returncode

def rsync_upload(local_path: Path, remote_dir: str):
    parts = _ssh_parts(
        SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if USE_DIRECT_FOR_RSYNC else None,
    )
    ssh_opt = " ".join(shlex.quote(p) for p in parts)
    cmd = (
        f"rsync -av --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{shlex.quote(str(local_path))} {SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY}:{shlex.quote(remote_dir)}/"
    )
    run_local(cmd)

def rsync_download(remote_path: str, local_dir: Path):
    parts = _ssh_parts(
        SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if USE_DIRECT_FOR_RSYNC else None,
    )
    ssh_opt = " ".join(shlex.quote(p) for p in parts)
    local_dir.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"rsync -av --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY}:{shlex.quote(remote_path)} {shlex.quote(str(local_dir))}/"
    )
    run_local(cmd)


## 2. 원격 기본 디렉토리 준비 및 점검

- 필요한 폴더 생성
- GPU/디스크 확인 (실패 시에도 진행되도록 `|| true`)
- `ssh` 접속 테스트까지 포함


In [3]:
# 접속 및 경로 생성
run_ssh(f"mkdir -p {REMOTE_RAW} {REMOTE_WORK} {REMOTE_CHECKPOINT} {REMOTE_LOG}")
run_ssh("echo 'SSH OK on $(hostname)' && nvidia-smi || true", check=False)
run_ssh("df -h . || true", check=False)


[ssh] $ mkdir -p ~/data/raw ~/data/work ~/data/work/checkpoints ~/logs
[ssh] $ echo 'SSH OK on $(hostname)' && nvidia-smi || true
SSH OK on $(hostname)
Thu Nov 27 02:02:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:0E:00.0 Off |                  Off |
| 30%   29C    P8             27W /  300W |       0MiB /  49140MiB |      0%    

0

## 3. 데이터 선택: mammary adenoma/adenocarcinoma 100개씩 (외장 2023)

- `mammary_adenoma_vs_adenocarcinoma_only(2023).parquet`에서 라벨을 읽어 두 클래스만 필터링
- `/Volumes/Expansion/2023/<FOLDER>/<FILE_NAME>.svs` 존재 여부 확인 후 클래스별 100개씩 셔플 샘플링
- 선택 결과는 `selected_slides` 리스트에 `slide_id/path/label` 필드로 저장


In [4]:
import random
import pandas as pd

SOURCE_SLIDE_ROOT = Path("/Volumes/Expansion/2023")
LABEL_PARQUET = Path("/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_only(2023).parquet")
SAMPLES_PER_CLASS = 100
LABEL_MAP = {"mammary_adenoma": 0, "mammary_adenocarcinoma": 1}

def _normalize_labels(df: pd.DataFrame) -> pd.DataFrame:
    label_norm = df["label"].astype(str).str.lower().str.strip()
    filtered = df.loc[label_norm.isin(LABEL_MAP.keys())].copy()
    filtered["label"] = label_norm.loc[filtered.index].map(LABEL_MAP)
    return filtered

def _collect(rows: pd.DataFrame, per_class: int):
    rows = rows.sample(frac=1, random_state=42).to_dict("records")
    picked = []
    for row in rows:
        slide_dir = SOURCE_SLIDE_ROOT / str(row["FOLDER"]).strip()
        file_names = [fn.strip() for fn in str(row["FILE_NAME"]).split("|") if fn.strip()]
        slide_id = row.get("INSP_RQST_NO", row.get("INSP_RQST_NUM", row["FOLDER"]))
        for file_name in file_names:
            fn = file_name if file_name.lower().endswith(".svs") else f"{file_name}.svs"
            slide_path = slide_dir / fn
            if slide_path.exists():
                picked.append({"slide_id": slide_id, "path": slide_path, "label": int(row["label"])})
                break
        if len(picked) >= per_class:
            break
    return picked

df_labels = _normalize_labels(pd.read_parquet(LABEL_PARQUET))
pos_rows = df_labels[df_labels["label"] == 1]
neg_rows = df_labels[df_labels["label"] == 0]
if pos_rows.empty or neg_rows.empty:
    raise ValueError(f"라벨 분포 확인 필요: pos={len(pos_rows)}, neg={len(neg_rows)}")

pos_pick = _collect(pos_rows, SAMPLES_PER_CLASS)
neg_pick = _collect(neg_rows, SAMPLES_PER_CLASS)
if len(pos_pick) < SAMPLES_PER_CLASS or len(neg_pick) < SAMPLES_PER_CLASS:
    print(f"경고: 요청 개수보다 부족 (pos {len(pos_pick)}/{SAMPLES_PER_CLASS}, neg {len(neg_pick)}/{SAMPLES_PER_CLASS})")

selected_slides = pos_pick[:SAMPLES_PER_CLASS] + neg_pick[:SAMPLES_PER_CLASS]
random.shuffle(selected_slides)

pos_cnt = sum(rec["label"] for rec in selected_slides)
neg_cnt = len(selected_slides) - pos_cnt
print(f"선정된 슬라이드: {len(selected_slides)}개 (pos={pos_cnt}, neg={neg_cnt})")
print("예시 3개:", selected_slides[:3])


선정된 슬라이드: 200개 (pos=100, neg=100)
예시 3개: [{'slide_id': '20230404-122-0002', 'path': PosixPath('/Volumes/Expansion/2023/S23-01921/S23-01921#1#A##0.svs'), 'label': 1}, {'slide_id': '20231102-138-0004', 'path': PosixPath('/Volumes/Expansion/2023/S23-07166/S23-07166#1###5.svs'), 'label': 1}, {'slide_id': '20231211-147-0001', 'path': PosixPath('/Volumes/Expansion/2023/S23-08693/S23-08693#1#A##1.svs'), 'label': 1}]


## 4. 원격 타일링 함수 (SVS 1개)

- RunPod에서 타일링 후 옵션에 따라 원본 삭제
- `TILING_CMD_TEMPLATE`를 gigapath에 맞게 수정


In [5]:
def remote_tile(svs_remote_path: str) -> str:
    svs_name = Path(svs_remote_path).name
    tile_dir = f"{REMOTE_WORK}/{Path(svs_name).stem}_tiles"
    cmd = TILING_CMD_TEMPLATE.format(
        input=shlex.quote(svs_remote_path),
        output=shlex.quote(tile_dir),
        tile_size=TILE_SIZE,
        overlap=TILE_OVERLAP,
    )
    run_ssh(cmd)
    if DELETE_SVS_AFTER_TILING:
        run_ssh(f"rm -f {shlex.quote(svs_remote_path)}")
        print(f"Deleted original svs: {svs_remote_path}")
    return tile_dir


## 5. 원격 벡터화 함수 (타일 디렉토리 1개)

- 타일 디렉토리를 받아 feature 파일 생성
- 완료 후 옵션에 따라 타일 삭제
- `FEATURE_CMD_TEMPLATE`를 gigapath에 맞게 수정


In [6]:
def remote_featurize(tile_dir: str) -> str:
    feat_path = f"{tile_dir}_feats.npy"
    cmd = FEATURE_CMD_TEMPLATE.format(
        tiles=shlex.quote(tile_dir),
        out=shlex.quote(feat_path),
    )
    run_ssh(cmd)
    if DELETE_TILES_AFTER_FEATURE:
        run_ssh(f"rm -rf {shlex.quote(tile_dir)}")
        print(f"Deleted tiles: {tile_dir}")
    print(f"Feature saved: {feat_path}")
    return feat_path


## 6. 선택 슬라이드 업로드→타일→벡터화 (순차 처리)

- `selected_slides`(각 클래스 최대 100개)를 순서대로 업로드/처리
- 슬라이드 1개 단위로 rsync 업로드 → 타일링 → 벡터화 → 옵션에 따라 원본/타일 삭제
- `RUN_SLIDE_PROCESSING` 토글로 실행 여부를 제어하고, `PROCESS_LIMIT`로 일부만 시범 실행 가능


In [11]:
def process_selected_slides(slides=None, limit=None):
    slides = slides or selected_slides
    if not slides:
        print("selected_slides가 비어있습니다. 3단계 데이터 선택 셀을 실행하세요.")
        return []

    to_run = slides if limit is None else slides[:limit]
    results = []
    for idx, rec in enumerate(to_run, 1):
        local_path = Path(rec["path"])
        remote_svs = f"{REMOTE_RAW}/{local_path.name}"
        print(f"\n[{idx}/{len(to_run)}] {local_path.name} (label={rec['label']})")
        rsync_upload(local_path, REMOTE_RAW)
        start = time.time()
        tile_dir = remote_tile(remote_svs)
        feat_path = remote_featurize(tile_dir)
        elapsed = time.time() - start
        results.append({
            **rec,
            "remote_svs": remote_svs,
            "feature_path": feat_path,
            "elapsed_min": elapsed / 60,
        })
        print(f"완료: {local_path.name} -> {feat_path} ({elapsed/60:.1f} min)")
    return results

RUN_SLIDE_PROCESSING = True
PROCESS_LIMIT = None  # 예: 10 으로 설정하면 앞 10개만 처리

if RUN_SLIDE_PROCESSING:
    processed_features = process_selected_slides(limit=PROCESS_LIMIT)
else:
    print("Processing skipped (RUN_SLIDE_PROCESSING=True 로 순차 실행)")



[1/200] S23-01921#1#A##0.svs (label=1)
[local] $ rsync -av --partial --progress -e 'ssh -i '"'"'~/.ssh/runpod_peter'"'"' -p 13458 root@216.81.151.15' '/Volumes/Expansion/2023/S23-01921/S23-01921#1#A##0.svs' root@216.81.151.15:'~/data/raw'/


bash: line 1: root@216.81.151.15: command not found
rsync(8278): error: unexpected end of file
rsync(8278): error: io_read_nonblocking
rsync(8278): error: io_read_buf
rsync(8278): error: io_read_int
rsync(8278): warning: child 8279 exited with status 127


RuntimeError: Local command failed: rsync -av --partial --progress -e 'ssh -i '"'"'~/.ssh/runpod_peter'"'"' -p 13458 root@216.81.151.15' '/Volumes/Expansion/2023/S23-01921/S23-01921#1#A##0.svs' root@216.81.151.15:'~/data/raw'/

## 7. 원격 학습 실행 (옵션)

- `RUN_TRAINING=True`로 토글
- `TRAIN_CMD_TEMPLATE`를 gigapath 학습 스크립트에 맞게 수정
- 체크포인트/로그 경로는 REMOTE_CHECKPOINT/REMOTE_LOG


In [9]:
RUN_TRAINING = False  # True로 바꾸면 학습 실행

def run_remote_training():
    train_cmd = TRAIN_CMD_TEMPLATE.format(
        features_dir=shlex.quote(REMOTE_WORK),
        ckpt_dir=shlex.quote(REMOTE_CHECKPOINT),
        log_dir=shlex.quote(REMOTE_LOG),
    )
    print(f"Training command:{train_cmd}")
    run_ssh(train_cmd)

if RUN_TRAINING:
    run_remote_training()
else:
    print("Training skipped (set RUN_TRAINING=True to run)")


Training skipped (set RUN_TRAINING=True to run)


## 8. 결과 다운로드/아카이브

- 필요 체크포인트/feature를 로컬로 rsync 다운로드
- 오래된 체크포인트는 RunPod에서 압축 후 삭제


In [10]:
def download_checkpoints(local_dir: Path):
    rsync_download(f"{REMOTE_CHECKPOINT}/", local_dir)

def remote_archive_and_delete(path_glob: str):
    # 예: path_glob="~/data/work/checkpoints/epoch*"
    cmd = f"for p in {path_glob}; do [ -e "$p" ] || continue; tar -czf ${p}.tar.gz -C $(dirname $p) $(basename $p) && rm -rf $p; done"
    run_ssh(cmd)


SyntaxError: invalid syntax (1354155642.py, line 6)

## 9. 추천 워크플로우 (셀 실행 순서)

1) 0~2 셀 실행: 설정/접속/경로 준비
2) 3 셀 실행: 외장 `/Volumes/Expansion/2023`에서 클래스별 100개 슬라이드 샘플링(`selected_slides` 확인)
3) 4~5 셀로 타일링/벡터화 함수 정의
4) 6 셀에서 `RUN_SLIDE_PROCESSING=True`로 설정 후 실행: 슬라이드별 업로드→타일→벡터 순차 처리
5) 필요시 7 셀에서 학습 실행 (`RUN_TRAINING=True`), 8 셀로 체크포인트/결과 다운로드

공간을 아끼려면 항상: 타일링 후 원본 삭제, 벡터화 후 타일 삭제, 체크포인트는 주기적으로 압축/정리.
